# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described and accessible via a Croissant schema hosted online.

In [ ]:
# Ensure `mlcroissant` (and pandas) are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display overview information
print(f"{getattr(metadata, 'name', '<no name>')}: {getattr(metadata, 'description', '<no description>')}")

## 2. Data Overview
List and explore all available record sets in the dataset, referencing by their `@id`.

In [ ]:
# List all record sets and their @id
def get_record_sets(md):
    try:
        # Try attribute (object style)
        return getattr(md, 'recordSet', [])
    except Exception:
        return []

record_sets = get_record_sets(metadata)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available record sets (@id):")
    for r in record_sets:
        print(f"- {getattr(r, '@id', str(r))}: {getattr(r, 'name', '<no name>')}")

### Find All Record Set IDs
Assign the relevant record set `@id` to a variable for further use.

In [ ]:
# For this dataset, the Croissant schema defines record sets dynamically. If none are found, print notice.
if not record_sets:
    # Try extracting the record set ids another way
    print("No recordSet found in metadata. Inspecting further ...")
    # mlcroissant dynamically constructs the record set for the tabular data usually,
    # using the CSV/TSV referenced by distribution. Let's inspect the resources:
    print("List all distribution @id (may correspond to files/record sets):")
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            print(f"- {getattr(dist, '@id', str(dist))}")

## Preview One Record Set

Fetch a preview sample from a specific record set (using the chosen `@id`). 

In [ ]:
# Since no recordSet was present above, let's list available dataset entrypoints by inspecting dataset._record_sets
print("Available record set @id in loaded dataset:")
for rs_id in dataset._record_sets:
    print(f"- {rs_id}")

# Select the first available record set for further exploration:
if dataset._record_sets:
    default_record_set_id = list(dataset._record_sets.keys())[0]
    print(f"\nUsing record set: {default_record_set_id}")

    # List available fields/column headers for this record set
    print("Available field @id in this record set:")
    for field in dataset._record_sets[default_record_set_id].fields:
        print(f"- {getattr(field, '@id', str(field))}: {getattr(field, 'name', '<no name>')}")

    # Show one sample record
    for rec in dataset.records(record_set=default_record_set_id):
        print("Sample record:", rec)
        break
else:
    print("No accessible record sets found in this dataset.")

## 3. Data Extraction

Load the data for each record set into a pandas DataFrame. Use the record set and field `@id`s only.

In [ ]:
dataframes = {}
record_set_ids = list(dataset._record_sets.keys())
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from record set {record_set_id}")

# Show columns of the main record set and preview
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore and process a numeric variable in the dataset, referencing column names by their `@id` only. We will:

* Select and filter records based on a numeric field.
* Normalize the selected field.
* (If possible) Group data by a categorical field to compute summary statistics.

**Note:** For this dataset, you may wish to adjust the `numeric_field_id` or `group_field_id` to match your actual column names as printed above (see previous output). The column/field names are typically their `@id` or property name from the Croissant schema.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id].copy()
print("Sample of column names:@id:", df.columns.tolist())

# Attempt to pick a numeric field based on typical clinical column ids
# Please update 'cr:Age_at_second_CRC' to the actual numeric @id if found in the columns
numeric_field_id = None
# Try to find a likely age or interval column
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower() or ('cr:' in col and 'Age' in col):
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

print(f"Using numeric field: {numeric_field_id}")

if numeric_field_id:
    # Convert column to numeric if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Compute a threshold for filtering (e.g., > median value)
    threshold = df[numeric_field_id].median() if pd.notnull(df[numeric_field_id]).any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    colnorm = f"{numeric_field_id}_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, colnorm]].head())

    # Try to group by a likely field (e.g., 'cr:Sex', 'cr:Location_of_second_CRC', etc.)
    group_field_id = None
    for col in df.columns:
        if ('sex' in col.lower()) or ('gender' in col.lower()) or ('location' in col.lower()) or ('cr:' in col and 'Sex' in col):
            group_field_id = col
            break
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions and relationships using matplotlib/seaborn, referencing variables by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plotting the distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is available, visualize by category
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated step-by-step data acquisition and exploration of the FAIR² clinical dataset using `mlcroissant`. The process included loading metadata, discovering available record sets (referenced exclusively by their `@id`), extracting fields, performing basic EDA, filtering and normalization, and generating visualizations. 

You can further adapt this notebook to perform dataset-specific analyses or to prepare data for machine learning workflows, always ensuring to reference entities by their Croissant `@id` for consistency and reproducibility.